In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from imblearn.over_sampling import SMOTE
import xgboost as xgb


In [4]:
data=pd.read_csv("Datasets/TRAIN_DATA.csv")
test_data=pd.read_csv("Datasets/TEST_DATA.csv")


In [5]:
print(len(data))
print(len(test_data))


630000
70000


In [6]:
data.head()

,id,age,alcohol_consumption_per_week,physical_activity_minutes_per_week,diet_score,sleep_hours_per_day,screen_time_hours_per_day,bmi,waist_to_hip_ratio,systolic_bp,...,gender,ethnicity,education_level,income_level,smoking_status,employment_status,family_history_diabetes,hypertension_history,cardiovascular_history,diagnosed_diabetes
0,0,55,2,89,4.7,7.6,6.8,30.7,0.91,131,...,Female,White,Graduate,Low,Never,Employed,0,0,0,1.0
1,1,50,1,59,7.1,4.9,8.6,27.1,0.87,113,...,Male,White,Graduate,Middle,Current,Employed,0,0,0,1.0
2,2,34,4,46,1.7,6.6,4.0,32.0,0.94,119,...,Female,White,Highschool,Lower-Middle,Current,Unemployed,0,0,0,0.0
3,3,69,2,245,3.9,6.0,3.5,30.1,0.92,135,...,Male,Black,Graduate,Lower-Middle,Never,Employed,0,0,0,0.0
4,4,44,2,62,6.4,7.9,5.6,22.5,0.83,104,...,Female,White,Graduate,Lower-Middle,Never,Employed,0,0,0,0.0


In [7]:
data.info()         
data.describe()  

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 630000 entries, 0 to 629999
Data columns (total 26 columns):
 #   Column                              Non-Null Count   Dtype  
---  ------                              --------------   -----  
 0   id                                  630000 non-null  int64  
 1   age                                 630000 non-null  int64  
 2   alcohol_consumption_per_week        630000 non-null  int64  
 3   physical_activity_minutes_per_week  630000 non-null  int64  
 4   diet_score                          630000 non-null  float64
 5   sleep_hours_per_day                 630000 non-null  float64
 6   screen_time_hours_per_day           630000 non-null  float64
 7   bmi                                 630000 non-null  float64
 8   waist_to_hip_ratio                  630000 non-null  float64
 9   systolic_bp                         630000 non-null  int64  
 10  diastolic_bp                        630000 non-null  int64  
 11  heart_rate                

,id,age,alcohol_consumption_per_week,physical_activity_minutes_per_week,diet_score,sleep_hours_per_day,screen_time_hours_per_day,bmi,waist_to_hip_ratio,systolic_bp,diastolic_bp,heart_rate,cholesterol_total,hdl_cholesterol,ldl_cholesterol,triglycerides,family_history_diabetes,hypertension_history,cardiovascular_history,diagnosed_diabetes
count,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000
mean,314999.500000,50.363227,2.071648,80.231557,5.963251,7.001907,6.012962,25.874561,0.858755,116.295073,75.440187,70.171635,186.826567,53.823516,102.914338,123.084843,0.149430,0.181995,0.030349,0.623295
std,181865.479132,11.652836,1.047572,51.201002,1.463775,0.901905,2.022679,2.860022,0.037975,11.008478,6.824568,6.938096,16.734801,8.269259,19.024615,24.749024,0.356512,0.385841,0.171546,0.484560
min,0.000000,19.000000,1.000000,1.000000,0.100000,3.100000,0.600000,15.100000,0.680000,91.000000,51.000000,42.000000,117.000000,21.000000,51.000000,31.000000,0.000000,0.000000,0.000000,0.000000
25%,157499.750000,42.000000,1.000000,49.000000,5.000000,6.400000,4.600000,23.900000,0.830000,108.000000,71.000000,65.000000,175.000000,48.000000,89.000000,106.000000,0.000000,0.000000,0.000000,0.000000
50%,314999.500000,50.000000,2.000000,71.000000,6.000000,7.000000,6.000000,25.900000,0.860000,116.000000,75.000000,70.000000,187.000000,54.000000,103.000000,123.000000,0.000000,0.000000,0.000000,1.000000
75%,472499.250000,58.000000,3.000000,96.000000,7.000000,7.600000,7.400000,27.800000,0.880000,124.000000,80.000000,75.000000,199.000000,59.000000,116.000000,139.000000,0.000000,0.000000,0.000000,1.000000
max,629999.000000,89.000000,9.000000,747.000000,9.900000,9.900000,16.500000,38.400000,1.050000,163.000000,104.000000,101.000000,289.000000,90.000000,205.000000,290.000000,1.000000,1.000000,1.000000,1.000000


In [8]:
data.isnull().sum()  
data.duplicated().sum()

np.int64(0)

# Drop id cause it's unique

In [9]:
datasample = data.drop(columns=['id'])
datasample.head()

test_data = test_data.drop(columns=['id'], errors='ignore')

# data visualization

# split data


In [10]:
X = datasample.drop('diagnosed_diabetes', axis=1)
y = datasample['diagnosed_diabetes']

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# convert target variable to integer type
datasample['diagnosed_diabetes'] = datasample['diagnosed_diabetes'].astype(int)
print(datasample['diagnosed_diabetes'].dtype)
print(datasample['diagnosed_diabetes'].value_counts(normalize=True))

int64
diagnosed_diabetes
1    0.623295
0    0.376705
Name: proportion, dtype: float64


In [11]:
print(len(X_train))
print(len(X_test))


504000
126000


In [12]:
print(X_train.columns)
print(X_train.head())


Index(['age', 'alcohol_consumption_per_week',
       'physical_activity_minutes_per_week', 'diet_score',
       'sleep_hours_per_day', 'screen_time_hours_per_day', 'bmi',
       'waist_to_hip_ratio', 'systolic_bp', 'diastolic_bp', 'heart_rate',
       'cholesterol_total', 'hdl_cholesterol', 'ldl_cholesterol',
       'triglycerides', 'gender', 'ethnicity', 'education_level',
       'income_level', 'smoking_status', 'employment_status',
       'family_history_diabetes', 'hypertension_history',
       'cardiovascular_history'],
      dtype='object')
        age  alcohol_consumption_per_week  physical_activity_minutes_per_week  \
337030   61                             2                                  53   
180977   38                             1                                  70   
490907   52                             2                                  92   
399622   51                             2                                  52   
622383   37                             5 

# feature engineering

In [13]:
def advanced_feature_engineering_final(df):
    df = df.copy()
    
    def has_col(col_name):
        return col_name in df.columns

    # ============ YOUR EXISTING FEATURES ============
    # Blood pressure
    if has_col('systolic_bp') and has_col('diastolic_bp'):
        df['pulse_pressure'] = df['systolic_bp'] - df['diastolic_bp']
        df['map'] = df['diastolic_bp'] + (df['systolic_bp'] - df['diastolic_bp']) / 3
        df['bp_risk'] = ((df['systolic_bp'] - 120) / 20) + ((df['diastolic_bp'] - 80) / 10)
        # NEW: Binary hypertension flag
        df['hypertension'] = ((df['systolic_bp'] >= 140) | (df['diastolic_bp'] >= 90)).astype(int)

    # Lipids
    if has_col('triglycerides') and has_col('hdl_cholesterol'):
        df['atherogenic_index'] = np.log((df['triglycerides'] + 1e-5) / (df['hdl_cholesterol'] + 1e-5))
        # NEW: Binary lipid flags
        df['high_triglycerides'] = (df['triglycerides'] >= 150).astype(int)
        df['low_hdl'] = (df['hdl_cholesterol'] < 40).astype(int)
        df['trig_hdl_ratio'] = df['triglycerides'] / (df['hdl_cholesterol'] + 1e-5)
    
    if has_col('cholesterol_total') and has_col('hdl_cholesterol'):
        df['ldl_hdl_ratio'] = (df['cholesterol_total'] - df['hdl_cholesterol']) / (df['hdl_cholesterol'] + 1e-5)
        df['high_total_chol'] = (df['cholesterol_total'] >= 200).astype(int)

    # Metabolic score
    if has_col('bmi') and has_col('waist_to_hip_ratio') and has_col('triglycerides') and has_col('hdl_cholesterol'):
        df['metabolic_score'] = (df['bmi']/25 + df['waist_to_hip_ratio']/0.85 + df['triglycerides']/150 - df['hdl_cholesterol']/50)
        # NEW: Metabolic syndrome binary
        df['metabolic_syndrome'] = (
            (df['bmi'] >= 30).astype(int) +
            (df['waist_to_hip_ratio'] >= 0.85).astype(int) +
            (df['triglycerides'] >= 150).astype(int) +
            (df['hdl_cholesterol'] < 40).astype(int)
        ).clip(upper=1)

    # Categories
    if has_col('bmi'):
        df['bmi_category'] = pd.cut(df['bmi'], bins=[0, 18.5, 25, 30, 35, 100], labels=[0, 1, 2, 3, 4]).astype(float)
        df['obese'] = (df['bmi'] >= 30).astype(int)
    
    if has_col('age'):
        df['age_category'] = pd.cut(df['age'], bins=[0, 30, 40, 50, 60, 100], labels=[0, 1, 2, 3, 4]).astype(float)
        df['age_over_50'] = (df['age'] > 50).astype(int)

    # Optional ratio
    if has_col('physical_activity_minutes_per_week') and has_col('screen_time_hours_per_day'):
        df['active_sedentary_ratio'] = (df['physical_activity_minutes_per_week'] / 7) / (df['screen_time_hours_per_day'] + 1)
        df['sedentary'] = (df['screen_time_hours_per_day'] > 5).astype(int)

    # ============ NEW: SIMPLE INTERACTIONS ============
    # These are NEW and CRITICAL
    
    # 1. Simple interaction
    if has_col('age') and has_col('bmi'):
        df['age_times_bmi'] = df['age'] * df['bmi'] / 100
    
    # 2. BP + Triglycerides
    if has_col('systolic_bp') and has_col('triglycerides'):
        df['bp_times_trig'] = df['systolic_bp'] * df['triglycerides'] / 1000
    
    # 3. Age + BP
    if has_col('age') and has_col('systolic_bp'):
        df['age_times_bp'] = df['age'] * df['systolic_bp'] / 1000
    
    # 4. Family history risk amplifier
    if has_col('family_history_diabetes') and has_col('age'):
        df['family_risk'] = df['family_history_diabetes'] * (df['age'] > 40).astype(int)
    
    return df

# Application
print("🔧 Application du Feature Engineering Avancé...")

X_train_fe_advanced = advanced_feature_engineering_final(X_train)
X_test_fe_advanced = advanced_feature_engineering_final(X_test)

print(f"✅ Avant: {X_train.shape}")
print(f"✅ Après: {X_train_fe_advanced.shape}")
print(f"   Nouvelles features créées: {X_train_fe_advanced.shape[1] - X_train.shape[1]}")

🔧 Application du Feature Engineering Avancé...
✅ Avant: (504000, 24)
✅ Après: (504000, 46)
   Nouvelles features créées: 22


In [14]:


print(f"\n📊 Shape avant: {X_train.shape}")
print(f"📊 Shape après: {X_train_fe_advanced.shape}")

# Check critical features
print(f"\n🔑 CHECKING NEW FEATURES:")
check_features = [
    'hypertension', 'high_triglycerides', 'low_hdl', 'obese',
    'age_over_50', 'age_times_bmi', 'bp_times_trig', 'metabolic_syndrome'
]

for feat in check_features:
    if feat in X_train_fe_advanced.columns:
        print(f"  ✅ {feat}: CREATED")
        # Show sample values
        print(f"     Sample: {X_train_fe_advanced[feat].head(3).values}")
    else:
        print(f"  ❌ {feat}: MISSING")

print("="*60)


📊 Shape avant: (504000, 24)
📊 Shape après: (504000, 46)

🔑 CHECKING NEW FEATURES:
  ✅ hypertension: CREATED
     Sample: [0 0 0]
  ✅ high_triglycerides: CREATED
     Sample: [0 0 0]
  ✅ low_hdl: CREATED
     Sample: [0 0 0]
  ✅ obese: CREATED
     Sample: [0 0 0]
  ✅ age_over_50: CREATED
     Sample: [1 0 1]
  ✅ age_times_bmi: CREATED
     Sample: [14.152  8.626 14.456]
  ✅ bp_times_trig: CREATED
     Sample: [12.3  14.08 13.13]
  ✅ metabolic_syndrome: CREATED
     Sample: [0 0 1]


# Preprocessing / scalling / encoding

In [15]:
print("⚙️  Preprocessing avec nouvelles features...")

# Colonnes catégorielles (les mêmes qu'avant)
categorical_cols = [
    'gender', 'ethnicity', 'education_level', 'income_level', 
    'smoking_status', 'employment_status', 'family_history_diabetes',
    'hypertension_history', 'cardiovascular_history'
]

categorical_cols_actual = [col for col in categorical_cols if col in X_train_fe_advanced.columns]
numeric_cols_actual = [col for col in X_train_fe_advanced.columns if col not in categorical_cols_actual]

print(f"   Colonnes numériques: {len(numeric_cols_actual)}")
print(f"   Colonnes catégorielles: {len(categorical_cols_actual)}")

# Pipeline
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

preprocessor_advanced = ColumnTransformer([
    ("num", numeric_pipeline, numeric_cols_actual),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols_actual)
], remainder='drop')

# 1️⃣ Fit and transform training data
X_train_processed_advanced = preprocessor_advanced.fit_transform(X_train_fe_advanced)
feature_names = preprocessor_advanced.get_feature_names_out()
X_train_processed_advanced = pd.DataFrame(X_train_processed_advanced, columns=feature_names) # type: ignore

# 2️⃣ Transform test data
X_test_processed_advanced = preprocessor_advanced.transform(X_test_fe_advanced)
X_test_processed_advanced = pd.DataFrame(X_test_processed_advanced, columns=feature_names) # type: ignore

# ✅ Now both train and test have the same columns and order


print(f"✅ Train processed: {X_train_processed_advanced.shape}")
print(f"✅ Test processed: {X_test_processed_advanced.shape}\n")

⚙️  Preprocessing avec nouvelles features...
   Colonnes numériques: 37
   Colonnes catégorielles: 9
✅ Train processed: (504000, 67)
✅ Test processed: (126000, 67)



# Data check

In [16]:
# Add this BEFORE any modeling
print("🔍 Checking data quality...")
print(f"Train shape: {X_train_processed_advanced.shape}")
print(f"Test shape: {X_test_processed_advanced.shape}")

# Check for NaN
print(f"\nNaN in train: {X_train_processed_advanced.isna().sum().sum()}") # type: ignore
print(f"NaN in test: {X_test_processed_advanced.isna().sum().sum()}") # type: ignore

# Check class balance
print(f"\nClass balance in y_train: {y_train.value_counts(normalize=True).round(3)}")
print(f"Class balance in y_test: {y_test.value_counts(normalize=True).round(3)}")

🔍 Checking data quality...
Train shape: (504000, 67)
Test shape: (126000, 67)

NaN in train: 0
NaN in test: 0

Class balance in y_train: diagnosed_diabetes
1.0    0.623
0.0    0.377
Name: proportion, dtype: float64
Class balance in y_test: diagnosed_diabetes
1.0    0.623
0.0    0.377
Name: proportion, dtype: float64


# no smote cause it's hurts lightboost

In [17]:
# print("⚖️  Application de SMOTE...")

# smote_advanced = SMOTE(random_state=42)
# X_train_balanced_advanced, y_train_balanced_advanced = smote_advanced.fit_resample(
#     X_train_processed_advanced, y_train
# )

# print(f"✅ Données balancées: {X_train_balanced_advanced.shape}")
# print(f"   Distribution: {pd.Series(y_train_balanced_advanced).value_counts(normalize=True).to_dict()}\n")


# lightGBM

In [18]:
# ================================================================
# FINAL ACCURACY-OPTIMIZED LIGHTGBM (NO SMOTE)
# ================================================================
import numpy as np
import lightgbm as lgb
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("🚀 FINAL MODEL — NO SMOTE — ACCURACY OPTIMIZED")
print("="*70)

# ================================================================
# TRAIN ON REAL DATA (THIS IS THE FIX 🔑)
# ================================================================
X_train_final = X_train_processed_advanced   # ✅ REAL DATA
y_train_final = y_train                      # ✅ REAL TARGET

# ================================================================
# LIGHTGBM CONFIG (ACCURACY-FOCUSED)
# ================================================================
lgb_acc = lgb.LGBMClassifier(
    n_estimators=350,
    max_depth=8,
    learning_rate=0.03,
    num_leaves=32,
    min_child_samples=30,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_alpha=0.5,
    reg_lambda=2.0,
    class_weight=None,      # 🚫 NO class_weight, NO SMOTE
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

print("⏳ Training LightGBM (real data)...")
lgb_acc.fit(X_train_final, y_train_final)
print("✅ Training completed")

# ================================================================
# PROBABILITIES
# ================================================================
y_proba_train = lgb_acc.predict_proba(X_train_final)[:, 1] # type: ignore
y_proba_test  = lgb_acc.predict_proba(X_test_processed_advanced)[:, 1] # type: ignore

# ================================================================
# MICRO THRESHOLD SEARCH (CRITICAL FOR +0.1%)
# ================================================================
best_acc = 0
best_threshold = 0.5

for t in np.arange(0.46, 0.52, 0.001):
    preds = (y_proba_test >= t).astype(int)
    acc = accuracy_score(y_test, preds)
    if acc > best_acc:
        best_acc = acc
        best_threshold = t

print(f"\n🎯 BEST THRESHOLD: {best_threshold:.3f}")
print(f"🔥 BEST TEST ACCURACY: {best_acc*100:.3f}%")

# ================================================================
# FINAL EVALUATION
# ================================================================
y_pred_train = (y_proba_train >= best_threshold).astype(int)
y_pred_test  = (y_proba_test  >= best_threshold).astype(int)

def evaluate(y_true, y_pred, name):
    print(f"\n📊 {name}")
    print(f"   Accuracy:  {accuracy_score(y_true, y_pred)*100:.2f}%")
    print(f"   Precision: {precision_score(y_true, y_pred)*100:.2f}%")
    print(f"   Recall:    {recall_score(y_true, y_pred)*100:.2f}%")
    print(f"   F1-score:  {f1_score(y_true, y_pred)*100:.2f}%")

evaluate(y_train_final, y_pred_train, "TRAIN")
evaluate(y_test, y_pred_test, "TEST")


🚀 FINAL MODEL — NO SMOTE — ACCURACY OPTIMIZED
⏳ Training LightGBM (real data)...
✅ Training completed

🎯 BEST THRESHOLD: 0.497
🔥 BEST TEST ACCURACY: 68.000%

📊 TRAIN
   Accuracy:  68.38%
   Precision: 70.24%
   Recall:    85.49%
   F1-score:  77.12%

📊 TEST
   Accuracy:  68.00%
   Precision: 69.96%
   Recall:    85.29%
   F1-score:  76.87%


# Check feature importance of data engeneering

In [19]:
# Check feature importance
importances = pd.DataFrame({
    'feature': X_train_final.columns,
    'importance': lgb_acc.feature_importances_
}).sort_values('importance', ascending=False)

print("\n🔍 Top 10 Most Important Features:")
print(importances.head(10))

print("\n🔍 Looking for your new features:")
new_feature_names = ['glucose_risk', 'age_bmi', 'bp_glucose']
for feat in new_feature_names:
    if feat in importances['feature'].values:
        rank = importiences[importances['feature'] == feat].index[0] + 1
        print(f"  {feat}: Rank #{rank}")
    else:
        print(f"  {feat}: NOT FOUND in features")


🔍 Top 10 Most Important Features:
                                    feature  importance
2   num__physical_activity_minutes_per_week        3752
14                       num__triglycerides        1086
0                                  num__age         550
33                       num__age_times_bmi         475
10                          num__heart_rate         422
11                   num__cholesterol_total         361
3                           num__diet_score         356
13                     num__ldl_cholesterol         348
6                                  num__bmi         338
7                   num__waist_to_hip_ratio         272

🔍 Looking for your new features:
  glucose_risk: NOT FOUND in features
  age_bmi: NOT FOUND in features
  bp_glucose: NOT FOUND in features


# submission

In [20]:
print("\n🎯 Preparing submission for TEST_DATA...")

# Feature engineering for test data# Feature engineering for test data
X_test_fe_advanced = advanced_feature_engineering_final(test_data)

# Fill missing columns with 0 (or NaN)
missing_cols = set(X_train_fe_advanced.columns) - set(X_test_fe_advanced.columns)
for col in missing_cols:
    X_test_fe_advanced[col] = 0  # or np.nan if you prefer

# Ensure same column order as training
X_test_fe_advanced = X_test_fe_advanced[X_train_fe_advanced.columns]

# Now transform
X_test_processed_advanced = preprocessor_advanced.transform(X_test_fe_advanced)

# Convert to DataFrame
feature_names = preprocessor_advanced.get_feature_names_out()
X_test_processed_advanced = pd.DataFrame(X_test_processed_advanced, columns=feature_names)


# Predict probabilities
y_proba_test_final = lgb_acc.predict_proba(X_test_processed_advanced)[:, 1]

# Apply the best threshold from validation
y_pred_test_final = (y_proba_test_final >= best_threshold).astype(float)

# Create submission DataFrame
submission = pd.DataFrame({
    'id': np.arange(1, len(y_pred_test_final) + 1),
    'diagnosed_diabetes': y_pred_test_final
})

# Save CSV
submission.to_csv('predictions.csv', index=False)
print(f"✅ Submission saved: {len(y_pred_test_final)} rows")
print(submission.head(15))
print("\n✅ FINAL CONFIGURATION SUMMARY")
print("="*40)


🎯 Preparing submission for TEST_DATA...
✅ Submission saved: 70000 rows
    id  diagnosed_diabetes
0    1                 1.0
1    2                 1.0
2    3                 1.0
3    4                 1.0
4    5                 0.0
5    6                 1.0
6    7                 1.0
7    8                 1.0
8    9                 1.0
9   10                 0.0
10  11                 1.0
11  12                 1.0
12  13                 1.0
13  14                 0.0
14  15                 1.0

✅ FINAL CONFIGURATION SUMMARY


# XGB

In [21]:
import xgboost as xgb


X_train_final = X_train_processed_advanced   # ✅ REAL DATA
y_train_final = y_train                      # ✅ REAL TARGET

xgb_acc = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.025,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_alpha=0.6,
    reg_lambda=2.0,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

print("⏳ Training XGBoost...")
xgb_acc.fit(X_train_processed_advanced, y_train)
print("✅ XGBoost trained")


# ================================================================
# PROBABILITIES
# ================================================================

y_proba_train_xgb = xgb_acc.predict_proba(X_train_final)[:, 1]
y_proba_test_xgb  = xgb_acc.predict_proba(X_test_processed_advanced)[:, 1]

# ================================================================
# MICRO THRESHOLD SEARCH (CRITICAL FOR +0.1%)
# ================================================================

best_acc_xgb = 0
best_threshold_xgb = 0.5

for t in np.arange(0.46, 0.52, 0.001):
    preds = (y_proba_test_xgb >= t).astype(int)
    acc = accuracy_score(y_test, preds)
    if acc > best_acc_xgb:
        best_acc_xgb = acc
        best_threshold_xgb = t

print(f"\n🎯 BEST THRESHOLD XGB: {best_threshold_xgb:.3f}")
print(f"🔥 BEST TEST ACCURACY XGB: {best_acc_xgb*100:.3f}%")

# ================================================================
# FINAL EVALUATION
# ================================================================

y_pred_train_xgb = (y_proba_train_xgb >= best_threshold_xgb).astype(int)
y_pred_test_xgb  = (y_proba_test_xgb  >= best_threshold_xgb).astype(int)

def evaluate_xgb(y_true, y_pred, name):
    print(f"\n📊 {name} XGB")
    print(f"   Accuracy:  {accuracy_score(y_true, y_pred)*100:.2f}%")
    print(f"   Precision: {precision_score(y_true, y_pred)*100:.2f}%")
    print(f"   Recall:    {recall_score(y_true, y_pred)*100:.2f}%")
    print(f"   F1-score:  {f1_score(y_true, y_pred)*100:.2f}%")

evaluate_xgb(y_train_final, y_pred_train_xgb, "TRAIN")
evaluate_xgb(y_test, y_pred_test_xgb, "TEST")


⏳ Training XGBoost...
✅ XGBoost trained


ValueError: Found input variables with inconsistent numbers of samples: [126000, 70000]

In [ ]:
# Average probabilities
y_proba_test_ensemble = (y_proba_test + y_proba_test_xgb) / 2
y_proba_train_ensemble = (y_proba_train + y_proba_train_xgb) / 2

# Threshold search
best_acc_ensemble = 0
best_threshold_ensemble = 0.5

for t in np.arange(0.46, 0.52, 0.001):
    y_pred_test_ens = (y_proba_test_ensemble >= t).astype(int)
    acc = accuracy_score(y_test, y_pred_test_ens)
    if acc > best_acc_ensemble:
        best_acc_ensemble = acc
        best_threshold_ensemble = t

print(f"🎯 BEST THRESHOLD ENSEMBLE: {best_threshold_ensemble:.3f}")
print(f"🔥 BEST TEST ACCURACY ENSEMBLE: {best_acc_ensemble*100:.3f}%")


🎯 BEST THRESHOLD ENSEMBLE: 0.498
🔥 BEST TEST ACCURACY ENSEMBLE: 68.007%


In [ ]:
print(f"y_test shape: {y_test.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"proba shape: {proba.shape}")

y_test shape: (126000,)
X_test shape: (70000, 67)
proba shape: (70000,)


In [ ]:
print("🔍 DATA INVENTORY:")
print("="*60)

# List all your datasets
datasets = {
    'X_train': X_train,
    'y_train': y_train,
    'X_test_70k': X_test,  # Your current 70K test features
    'X_test_processed_70k': X_test_processed_advanced,  # Processed 70K
    'y_test_full': y_test  # Your 126K test labels
}

for name, data in datasets.items():
    if isinstance(data, (np.ndarray, pd.DataFrame, pd.Series)):
        print(f"{name:25s}: {data.shape}")
    else:
        print(f"{name:25s}: Not available or wrong type")

🔍 DATA INVENTORY:
X_train                  : (504000, 67)
y_train                  : (504000,)
X_test_70k               : (70000, 67)
X_test_processed_70k     : (70000, 67)
y_test_full              : (126000,)


In [ ]:
# ================================================================
# ULTRA-OPTIMIZED LIGHTGBM
# ================================================================
import numpy as np
import lightgbm as lgb
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

print("🚀 ULTRA-OPTIMIZED LIGHTGBM - PUSHING FOR 68.5%+")
print("="*70)

# ================================================================
# OPTIMIZED CONFIGURATIONS
# ================================================================

configs = {
    "V1 - Balanced Precision/Recall": {
        'n_estimators': 500,
        'max_depth': 9,
        'learning_rate': 0.025,
        'num_leaves': 45,
        'min_child_samples': 25,
        'subsample': 0.85,
        'colsample_bytree': 0.85,
        'reg_alpha': 0.3,
        'reg_lambda': 1.5,
        'min_split_gain': 0.01,
        'min_child_weight': 0.001,
    },
    "V2 - High Accuracy Focus": {
        'n_estimators': 600,
        'max_depth': 7,
        'learning_rate': 0.018,
        'num_leaves': 35,
        'min_child_samples': 35,
        'subsample': 0.88,
        'colsample_bytree': 0.88,
        'reg_alpha': 0.5,
        'reg_lambda': 2.0,
        'min_split_gain': 0.02,
        'min_child_weight': 0.005,
    },
    "V3 - Complex Patterns": {
        'n_estimators': 450,
        'max_depth': 11,
        'learning_rate': 0.03,
        'num_leaves': 65,
        'min_child_samples': 15,
        'subsample': 0.82,
        'colsample_bytree': 0.82,
        'reg_alpha': 0.2,
        'reg_lambda': 1.0,
        'min_split_gain': 0.005,
        'min_child_weight': 0.001,
    },
    "V4 - Conservative": {
        'n_estimators': 700,
        'max_depth': 6,
        'learning_rate': 0.015,
        'num_leaves': 25,
        'min_child_samples': 40,
        'subsample': 0.9,
        'colsample_bytree': 0.9,
        'reg_alpha': 0.8,
        'reg_lambda': 3.0,
        'min_split_gain': 0.03,
        'min_child_weight': 0.01,
    }
}

# ================================================================
# TRAIN ALL CONFIGS
# ================================================================

models = {}
probas = {}

print("\n🔧 Training optimized configurations...")
print("-"*70)

for name, params in configs.items():
    print(f"\n⏳ {name}...")
    
    model = lgb.LGBMClassifier(
        **params,
        random_state=42,
        n_jobs=-1,
        verbose=-1,
        force_row_wise=True  # Better for large datasets
    )
    
    # Train with early stopping on validation
    from sklearn.model_selection import train_test_split
    X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
        X_train_processed_advanced, y_train, 
        test_size=0.1, 
        random_state=42,
        stratify=y_train
    )
    
    model.fit(
        X_train_split, y_train_split,
        eval_set=[(X_val_split, y_val_split)],
        eval_metric='binary_logloss',
        callbacks=[
            lgb.early_stopping(50, verbose=False),
            lgb.log_evaluation(0)
        ]
    )
    
    # Predict
    proba_test = model.predict_proba(X_test_processed_advanced)[:, 1]
    
    models[name] = model
    probas[name] = proba_test
    
    print(f"   ✅ Trained with {model.best_iteration_} iterations")

# ================================================================
# THRESHOLD OPTIMIZATION FOR EACH MODEL
# ================================================================

print("\n" + "="*70)
print("🎯 THRESHOLD OPTIMIZATION")
print("-"*70)

threshold_results = {}

for name, proba in probas.items():
    best_acc = 0
    best_t = 0.5
    
    # WIDER and FINER search
    for t in np.arange(0.40, 0.65, 0.0005):  # Much wider, finer search
        preds = (proba >= t).astype(int)
        acc = accuracy_score(y_test[:70000], preds)  # Use first 70K labels
        
        if acc > best_acc:
            best_acc = acc
            best_t = t
    
    threshold_results[name] = (best_t, best_acc)
    
    print(f"{name:25s}: Threshold={best_t:.4f}, Accuracy={best_acc*100:.4f}%")

# ================================================================
# ENSEMBLE STRATEGIES
# ================================================================

print("\n" + "="*70)
print("🤝 ENSEMBLE STRATEGIES")
print("-"*70)

# Get best 2 models
sorted_models = sorted(threshold_results.items(), key=lambda x: x[1][1], reverse=True)
best_model_name = sorted_models[0][0]
second_best_name = sorted_models[1][0]

print(f"Best model: {best_model_name}")
print(f"Second best: {second_best_name}")

# Strategy 1: Simple average of all
print("\n1. Simple average of all models:")
all_probas = np.array([probas[name] for name in configs.keys()])
avg_proba = np.mean(all_probas, axis=0)

best_acc_avg = 0
best_t_avg = 0.5
for t in np.arange(0.40, 0.65, 0.0005):
    preds = (avg_proba >= t).astype(int)
    acc = accuracy_score(y_test[:70000], preds)
    if acc > best_acc_avg:
        best_acc_avg = acc
        best_t_avg = t

print(f"   Threshold: {best_t_avg:.4f}, Accuracy: {best_acc_avg*100:.4f}%")

# Strategy 2: Weighted average of top 2
print("\n2. Weighted average of top 2 models:")
best_acc_weighted = 0
best_w = 0.5
best_t_weighted = 0.5

proba1 = probas[best_model_name]
proba2 = probas[second_best_name]

for w in np.arange(0.3, 0.8, 0.05):  # Try different weights
    weighted_proba = w * proba1 + (1-w) * proba2
    
    for t in np.arange(0.40, 0.65, 0.0005):
        preds = (weighted_proba >= t).astype(int)
        acc = accuracy_score(y_test[:70000], preds)
        if acc > best_acc_weighted:
            best_acc_weighted = acc
            best_w = w
            best_t_weighted = t

print(f"   Weight: {best_w:.2f} on {best_model_name}")
print(f"   Threshold: {best_t_weighted:.4f}, Accuracy: {best_acc_weighted*100:.4f}%")

# Strategy 3: Stack with Logistic Regression
print("\n3. Stacking with Logistic Regression:")
from sklearn.linear_model import LogisticRegression

# Create meta-features
meta_features = np.column_stack([probas[name] for name in configs.keys()])

# Split for meta-training
X_meta_train, X_meta_test, y_meta_train, y_meta_test = train_test_split(
    meta_features, y_test[:70000], 
    test_size=0.3, 
    random_state=42
)

stacker = LogisticRegression(C=0.1, max_iter=1000, random_state=42)
stacker.fit(X_meta_train, y_meta_train)

stacked_proba = stacker.predict_proba(meta_features)[:, 1]

best_acc_stacked = 0
best_t_stacked = 0.5
for t in np.arange(0.40, 0.65, 0.0005):
    preds = (stacked_proba >= t).astype(int)
    acc = accuracy_score(y_test[:70000], preds)
    if acc > best_acc_stacked:
        best_acc_stacked = acc
        best_t_stacked = t

print(f"   Threshold: {best_t_stacked:.4f}, Accuracy: {best_acc_stacked*100:.4f}%")

# ================================================================
# FINAL COMPARISON
# ================================================================

print("\n" + "="*70)
print("📊 FINAL RESULTS")
print("="*70)

results = {
    "Your original (68.00%)": 0.6800,
    f"Best single ({best_model_name})": threshold_results[best_model_name][1],
    "Average of all 4": best_acc_avg,
    "Weighted top 2": best_acc_weighted,
    "Stacked ensemble": best_acc_stacked
}

# Sort by accuracy
sorted_results = sorted(results.items(), key=lambda x: x[1], reverse=True)

print("\n🏆 RANKING:")
for i, (method, acc) in enumerate(sorted_results, 1):
    improvement = (acc - 0.6800) * 100
    marker = "🥇" if i == 1 else f"{i}."
    print(f"{marker} {method:25s}: {acc*100:.4f}% (Δ{improvement:+.4f}%)")

# ================================================================
# TRAIN FINAL ENSEMBLE ON FULL DATA
# ================================================================

print("\n" + "="*70)
print("🚀 TRAINING FINAL ENSEMBLE ON FULL DATA")
print("-"*70)

# Use best strategy
if sorted_results[0][0] == "Stacked ensemble":
    print("Training final stacked ensemble on full data...")
    
    # Retrain stacker on full meta-features
    stacker_final = LogisticRegression(C=0.1, max_iter=1000, random_state=42)
    stacker_final.fit(meta_features, y_test[:70000])
    
    final_proba = stacker_final.predict_proba(meta_features)[:, 1]
    final_threshold = best_t_stacked
    
elif sorted_results[0][0] == "Weighted top 2":
    print(f"Using weighted ensemble ({best_w:.2f}/{1-best_w:.2f})...")
    final_proba = best_w * proba1 + (1-best_w) * proba2
    final_threshold = best_t_weighted
    
else:
    print(f"Using best single model: {best_model_name}...")
    final_proba = probas[best_model_name]
    final_threshold = threshold_results[best_model_name][0]

# Final predictions
final_preds = (final_proba >= final_threshold).astype(int)

print(f"\n✅ Final threshold: {final_threshold:.4f}")
print(f"🔥 Final accuracy: {accuracy_score(y_test[:70000], final_preds)*100:.4f}%")
print(f"📊 Class distribution: {np.bincount(final_preds)}")
print(f"📈 Positive rate: {final_preds.mean()*100:.2f}%")

🚀 ULTRA-OPTIMIZED LIGHTGBM - PUSHING FOR 68.5%+

🔧 Training optimized configurations...
----------------------------------------------------------------------

⏳ V1 - Balanced Precision/Recall...
   ✅ Trained with 500 iterations

⏳ V2 - High Accuracy Focus...
   ✅ Trained with 600 iterations

⏳ V3 - Complex Patterns...
   ✅ Trained with 447 iterations

⏳ V4 - Conservative...
   ✅ Trained with 700 iterations

🎯 THRESHOLD OPTIMIZATION
----------------------------------------------------------------------
V1 - Balanced Precision/Recall: Threshold=0.4000, Accuracy=59.4714%
V2 - High Accuracy Focus : Threshold=0.4000, Accuracy=59.7300%
V3 - Complex Patterns    : Threshold=0.4000, Accuracy=59.4371%
V4 - Conservative        : Threshold=0.4005, Accuracy=59.8529%

🤝 ENSEMBLE STRATEGIES
----------------------------------------------------------------------
Best model: V4 - Conservative
Second best: V2 - High Accuracy Focus

1. Simple average of all models:
   Threshold: 0.4000, Accuracy: 59.6571

MemoryError: Unable to allocate 547. KiB for an array with shape (70000,) and data type float64